# 05 — Gold Aggregations

**Week:** 7

## Trusted Silver to Gold

Goal: Create project-specific Gold metric tables from approved Trusted Silver inputs.

This notebook defines KPI contracts, builds Gold Delta tables, validates their grain and measures, reconciles the outputs with the eligible Trusted Silver population, and performs controlled rerun checks.

**Week 7 boundary:** Gold design and build only. Power BI, dashboard export, and streaming are not implemented here.


## 1. Gold KPI Register

### KPI 1 — Monthly Collision Activity
- **Business question:** How does collision activity vary by month and urban/rural setting?
- **Gold table:** `gold_collision_monthly_summary`
- **Grain:** One row per `collision_year + collision_month + urban_rural_group`
- **Primary measure:** `collision_count = COUNT(*)`
- **Trusted Silver input:** `trusted_silver_collisions`
- **Scope:** `collision_index IS NOT NULL`

### KPI 2 — Vehicle Involvement by Vehicle Type
- **Business question:** How many vehicle involvements occur by vehicle type and year?
- **Gold table:** `gold_vehicle_type_summary`
- **Grain:** One row per `collision_year + vehicle_type`
- **Primary measure:** `vehicle_involvement_count = COUNT(*)`
- **Trusted Silver input:** `trusted_silver_vehicles`
- **Scope:** `vehicle_key IS NOT NULL`

### KPI 3 — Casualty Count by Severity and Age Group
- **Business question:** How are casualties distributed across severity and age groups?
- **Gold table:** `gold_casualty_severity_summary`
- **Grain:** One row per `collision_year + casualty_age_group + casualty_severity`
- **Primary measure:** `casualty_count = COUNT(*)`
- **Trusted Silver input:** `trusted_silver_casualties`
- **Scope:** `casualty_key IS NOT NULL`


In [ ]:
%sql
-- Verify approved Trusted Silver inputs

SHOW TABLES LIKE 'trusted_silver_*'


## 2. KPI 1 — Monthly Collision Activity

### KPI Contract

**Formula:**
- `collision_count = COUNT(*)`
- `severe_collision_count = SUM(CASE WHEN severe_collision_flag = 1 THEN 1 ELSE 0 END)`
- `severe_collision_rate = 100 × severe_collision_count / collision_count`

**Grain:** `collision_year + collision_month + urban_rural_group`

**Input:** `trusted_silver_collisions`

**Eligible rows:** `collision_index IS NOT NULL`

**Joins:** None.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_collision_monthly_summary
USING DELTA
AS
SELECT
    collision_year,
    collision_month,
    urban_rural_group,
    COUNT(*) AS collision_count,
    SUM(
        CASE
            WHEN severe_collision_flag = 1 THEN 1
            ELSE 0
        END
    ) AS severe_collision_count,
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN severe_collision_flag = 1 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS severe_collision_rate,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_collisions
WHERE collision_index IS NOT NULL
GROUP BY
    collision_year,
    collision_month,
    urban_rural_group


In [ ]:
%sql
SELECT *
FROM gold_collision_monthly_summary
ORDER BY collision_year, collision_month, urban_rural_group


In [ ]:
%sql
-- KPI 1: grain validation

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT_WS(
        '||',
        CAST(collision_year AS STRING),
        CAST(collision_month AS STRING),
        COALESCE(urban_rural_group, '<NULL>')
    )) AS distinct_grain_keys
FROM gold_collision_monthly_summary


In [ ]:
%sql
-- KPI 1: null key validation

SELECT
    SUM(CASE WHEN collision_year IS NULL THEN 1 ELSE 0 END) AS null_year,
    SUM(CASE WHEN collision_month IS NULL THEN 1 ELSE 0 END) AS null_month,
    SUM(CASE WHEN urban_rural_group IS NULL THEN 1 ELSE 0 END) AS null_urban_rural_group
FROM gold_collision_monthly_summary


In [ ]:
%sql
-- KPI 1: measure validation

SELECT
    SUM(CASE WHEN collision_count < 0 THEN 1 ELSE 0 END) AS negative_collision_count,
    SUM(CASE WHEN severe_collision_count < 0 THEN 1 ELSE 0 END) AS negative_severe_count,
    SUM(CASE WHEN severe_collision_rate < 0 OR severe_collision_rate > 100 THEN 1 ELSE 0 END) AS invalid_severe_rate
FROM gold_collision_monthly_summary


In [ ]:
%sql
-- KPI 1: measure reconciliation

SELECT
    (SELECT SUM(collision_count)
     FROM gold_collision_monthly_summary) AS gold_collision_total,

    (SELECT COUNT(*)
     FROM trusted_silver_collisions
     WHERE collision_index IS NOT NULL) AS eligible_collision_total


In [ ]:
%sql
-- KPI 1: scope reconciliation

SELECT
    COUNT(*) AS trusted_input_rows,
    SUM(CASE WHEN collision_index IS NOT NULL THEN 1 ELSE 0 END) AS eligible_rows,
    SUM(CASE WHEN collision_index IS NULL THEN 1 ELSE 0 END) AS outside_kpi_scope
FROM trusted_silver_collisions


In [ ]:
%sql
-- KPI 1: Delta table details

DESCRIBE DETAIL gold_collision_monthly_summary


In [ ]:
%sql
-- KPI 1: capture business rows before controlled rerun

CREATE OR REPLACE TEMP VIEW gold_collision_run_1 AS
SELECT
    collision_year,
    collision_month,
    urban_rural_group,
    collision_count,
    severe_collision_count,
    severe_collision_rate
FROM gold_collision_monthly_summary


In [ ]:
%sql
-- KPI 1: controlled rerun

CREATE OR REPLACE TABLE gold_collision_monthly_summary
USING DELTA
AS
SELECT
    collision_year,
    collision_month,
    urban_rural_group,
    COUNT(*) AS collision_count,
    SUM(CASE WHEN severe_collision_flag = 1 THEN 1 ELSE 0 END) AS severe_collision_count,
    ROUND(
        100.0 * SUM(CASE WHEN severe_collision_flag = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS severe_collision_rate,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_collisions
WHERE collision_index IS NOT NULL
GROUP BY collision_year, collision_month, urban_rural_group


In [ ]:
%sql
-- KPI 1: two-way business-row comparison

SELECT COUNT(*) AS differences
FROM (
    SELECT collision_year, collision_month, urban_rural_group,
           collision_count, severe_collision_count, severe_collision_rate
    FROM gold_collision_run_1
    EXCEPT
    SELECT collision_year, collision_month, urban_rural_group,
           collision_count, severe_collision_count, severe_collision_rate
    FROM gold_collision_monthly_summary
)


In [ ]:
%sql
SELECT COUNT(*) AS differences
FROM (
    SELECT collision_year, collision_month, urban_rural_group,
           collision_count, severe_collision_count, severe_collision_rate
    FROM gold_collision_monthly_summary
    EXCEPT
    SELECT collision_year, collision_month, urban_rural_group,
           collision_count, severe_collision_count, severe_collision_rate
    FROM gold_collision_run_1
)


## 3. KPI 2 — Vehicle Involvement by Vehicle Type

### KPI Contract

**Formula:** `vehicle_involvement_count = COUNT(*)`

**Grain:** `collision_year + vehicle_type`

**Input:** `trusted_silver_vehicles`

**Eligible rows:** `vehicle_key IS NOT NULL`

**Joins:** None.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_vehicle_type_summary
USING DELTA
AS
SELECT
    collision_year,
    vehicle_type,
    COUNT(*) AS vehicle_involvement_count,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_vehicles
WHERE vehicle_key IS NOT NULL
GROUP BY collision_year, vehicle_type


In [ ]:
%sql
SELECT *
FROM gold_vehicle_type_summary
ORDER BY collision_year, vehicle_type


In [ ]:
%sql
-- KPI 2: grain validation

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT_WS(
        '||',
        CAST(collision_year AS STRING),
        CAST(vehicle_type AS STRING)
    )) AS distinct_grain_keys
FROM gold_vehicle_type_summary


In [ ]:
%sql
-- KPI 2: measure validation

SELECT
    SUM(CASE WHEN vehicle_involvement_count < 0 THEN 1 ELSE 0 END) AS negative_counts
FROM gold_vehicle_type_summary


In [ ]:
%sql
-- KPI 2: measure reconciliation

SELECT
    (SELECT SUM(vehicle_involvement_count)
     FROM gold_vehicle_type_summary) AS gold_vehicle_total,

    (SELECT COUNT(*)
     FROM trusted_silver_vehicles
     WHERE vehicle_key IS NOT NULL) AS eligible_vehicle_total


In [ ]:
%sql
-- KPI 2: scope reconciliation

SELECT
    COUNT(*) AS trusted_input_rows,
    SUM(CASE WHEN vehicle_key IS NOT NULL THEN 1 ELSE 0 END) AS eligible_rows,
    SUM(CASE WHEN vehicle_key IS NULL THEN 1 ELSE 0 END) AS outside_kpi_scope
FROM trusted_silver_vehicles


In [ ]:
%sql
-- KPI 2: Delta table details

DESCRIBE DETAIL gold_vehicle_type_summary


In [ ]:
%sql
-- KPI 2: capture business rows before controlled rerun

CREATE OR REPLACE TEMP VIEW gold_vehicle_run_1 AS
SELECT collision_year, vehicle_type, vehicle_involvement_count
FROM gold_vehicle_type_summary


In [ ]:
%sql
-- KPI 2: controlled rerun

CREATE OR REPLACE TABLE gold_vehicle_type_summary
USING DELTA
AS
SELECT
    collision_year,
    vehicle_type,
    COUNT(*) AS vehicle_involvement_count,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_vehicles
WHERE vehicle_key IS NOT NULL
GROUP BY collision_year, vehicle_type


In [ ]:
%sql
-- KPI 2: two-way business-row comparison

SELECT COUNT(*) AS differences
FROM (
    SELECT collision_year, vehicle_type, vehicle_involvement_count
    FROM gold_vehicle_run_1
    EXCEPT
    SELECT collision_year, vehicle_type, vehicle_involvement_count
    FROM gold_vehicle_type_summary
)


In [ ]:
%sql
SELECT COUNT(*) AS differences
FROM (
    SELECT collision_year, vehicle_type, vehicle_involvement_count
    FROM gold_vehicle_type_summary
    EXCEPT
    SELECT collision_year, vehicle_type, vehicle_involvement_count
    FROM gold_vehicle_run_1
)


## 4. KPI 3 — Casualty Count by Severity and Age Group

### KPI Contract

**Formula:** `casualty_count = COUNT(*)`

**Grain:** `collision_year + casualty_age_group + casualty_severity`

**Input:** `trusted_silver_casualties`

**Eligible rows:** `casualty_key IS NOT NULL`

**Joins:** None.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_casualty_severity_summary
USING DELTA
AS
SELECT
    collision_year,
    casualty_age_group,
    casualty_severity,
    COUNT(*) AS casualty_count,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_casualties
WHERE casualty_key IS NOT NULL
GROUP BY collision_year, casualty_age_group, casualty_severity


In [ ]:
%sql
SELECT *
FROM gold_casualty_severity_summary
ORDER BY collision_year, casualty_age_group, casualty_severity


In [ ]:
%sql
-- KPI 3: grain validation

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT_WS(
        '||',
        CAST(collision_year AS STRING),
        CAST(casualty_age_group AS STRING),
        CAST(casualty_severity AS STRING)
    )) AS distinct_grain_keys
FROM gold_casualty_severity_summary


In [ ]:
%sql
-- KPI 3: measure validation

SELECT
    SUM(CASE WHEN casualty_count < 0 THEN 1 ELSE 0 END) AS negative_casualty_counts
FROM gold_casualty_severity_summary


In [ ]:
%sql
-- KPI 3: measure reconciliation

SELECT
    (SELECT SUM(casualty_count)
     FROM gold_casualty_severity_summary) AS gold_casualty_total,

    (SELECT COUNT(*)
     FROM trusted_silver_casualties
     WHERE casualty_key IS NOT NULL) AS eligible_casualty_total


In [ ]:
%sql
-- KPI 3: scope reconciliation

SELECT
    COUNT(*) AS trusted_input_rows,
    SUM(CASE WHEN casualty_key IS NOT NULL THEN 1 ELSE 0 END) AS eligible_rows,
    SUM(CASE WHEN casualty_key IS NULL THEN 1 ELSE 0 END) AS outside_kpi_scope
FROM trusted_silver_casualties


In [ ]:
%sql
-- KPI 3: Delta table details

DESCRIBE DETAIL gold_casualty_severity_summary


In [ ]:
%sql
-- KPI 3: capture business rows before controlled rerun

CREATE OR REPLACE TEMP VIEW gold_casualty_run_1 AS
SELECT
    collision_year,
    casualty_age_group,
    casualty_severity,
    casualty_count
FROM gold_casualty_severity_summary


In [ ]:
%sql
-- KPI 3: controlled rerun

CREATE OR REPLACE TABLE gold_casualty_severity_summary
USING DELTA
AS
SELECT
    collision_year,
    casualty_age_group,
    casualty_severity,
    COUNT(*) AS casualty_count,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_casualties
WHERE casualty_key IS NOT NULL
GROUP BY collision_year, casualty_age_group, casualty_severity


In [ ]:
%sql
-- KPI 3: two-way business-row comparison

SELECT COUNT(*) AS differences
FROM (
    SELECT collision_year, casualty_age_group, casualty_severity, casualty_count
    FROM gold_casualty_run_1
    EXCEPT
    SELECT collision_year, casualty_age_group, casualty_severity, casualty_count
    FROM gold_casualty_severity_summary
)


In [ ]:
%sql
SELECT COUNT(*) AS differences
FROM (
    SELECT collision_year, casualty_age_group, casualty_severity, casualty_count
    FROM gold_casualty_severity_summary
    EXCEPT
    SELECT collision_year, casualty_age_group, casualty_severity, casualty_count
    FROM gold_casualty_run_1
)


## 5. Final Gold Catalog

| Gold Table | Grain | Primary KPI |
|---|---|---|
| `gold_collision_monthly_summary` | Year + Month + Urban/Rural | Monthly Collision Count |
| `gold_vehicle_type_summary` | Year + Vehicle Type | Vehicle Involvement Count |
| `gold_casualty_severity_summary` | Year + Age Group + Severity | Casualty Count |


In [ ]:
%sql
-- Final Gold output verification

SHOW TABLES LIKE 'gold_*'


## 6. Week 7 Completion Boundary

This notebook contains:

- Project-specific KPI contracts
- Trusted Silver input verification
- Explicit KPI scope
- Gold Delta table construction
- Grain validation
- Measure validation
- Measure reconciliation
- Scope reconciliation
- Delta table inspection
- Controlled rerun comparison

Power BI, dashboard export, and streaming are outside the Week 7 implementation boundary.
